# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sehreen-Atta/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Mid-panel warehouse slice (`month=2026-03`), aggregated to one row per content item. No label-derived fields, no future windows, no product flags.

In [3]:
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

ROOT = Path("..")  # repo root when cwd is work/notebooks
if not (ROOT / ".env").exists():
    ROOT = Path("..") if (Path("..") / ".env").exists() else Path(".")
if not (ROOT / ".env").exists() and (Path("..") / ".." / ".env").exists():
    ROOT = Path("..") / ".."

env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Set HF_TOKEN in .env at the repo root")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
content_src = f"read_parquet('{REL}/dim_content.parquet')"
perf_src = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

SNAPSHOT = pd.Timestamp("2026-03-31")
STALE_DAYS = 180
MIN_IMPRESSIONS = 100

raw = con.sql(f"""
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        c.last_optimized_date,
        c.content_created_date,
        SUM(p.gsc_impressions) AS impressions,
        SUM(p.gsc_clicks) AS clicks,
        AVG(CASE WHEN p.gsc_avg_position > 0 THEN p.gsc_avg_position END) AS avg_position
    FROM {perf_src} p
    JOIN {content_src} c ON p.content_hash_id = c.content_hash_id
    WHERE p.gsc_data_available IS TRUE
    GROUP BY 1, 2, 3, 4
""").df()

raw["last_optimized_date"] = pd.to_datetime(raw["last_optimized_date"], errors="coerce")
raw["content_created_date"] = pd.to_datetime(raw["content_created_date"], errors="coerce")
ref_date = raw["last_optimized_date"].fillna(raw["content_created_date"])
raw["staleness_days"] = (SNAPSHOT - ref_date).dt.days
raw["ctr_pct"] = raw["clicks"] / raw["impressions"].replace(0, np.nan) * 100

work = raw[
    (raw["impressions"] >= MIN_IMPRESSIONS)
    & (raw["avg_position"].notna())
    & (raw["avg_position"] > 0)
    & (raw["staleness_days"].notna())
    & (raw["staleness_days"] >= 0)
].copy()

print(f"March 2026 content rows (impressions >= {MIN_IMPRESSIONS}, valid position & staleness): {len(work):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 content rows (impressions >= 100, valid position & staleness): 63,661


## Signal 1 — Staleness vs CTR

FlyRank's refresh flags assume older content tends to underperform on CTR. Bucket `staleness_days` (days since `last_optimized_date`, falling back to `content_created_date`) and compare average CTR (×100 percentages, March 2026 impressions ≥ 100).

In [4]:
def staleness_bucket(days: float) -> str:
    if days < 30:
        return "0-29 days"
    if days < 90:
        return "30-89 days"
    if days < 180:
        return "90-179 days"
    if days < 365:
        return "180-364 days"
    return "365+ days"

work["staleness_bucket"] = work["staleness_days"].map(staleness_bucket)
staleness_order = ["0-29 days", "30-89 days", "90-179 days", "180-364 days", "365+ days"]

signal1 = (
    work.groupby("staleness_bucket", observed=True)
    .agg(n=("content_hash_id", "count"), avg_ctr_pct=("ctr_pct", "mean"))
    .reindex(staleness_order)
    .reset_index()
)
display(signal1.assign(avg_ctr_pct=signal1["avg_ctr_pct"].round(4)))

,staleness_bucket,n,avg_ctr_pct
0,0-29 days,7016,0.3892
1,30-89 days,16432,0.2438
2,90-179 days,10277,0.1970
3,180-364 days,21030,0.2069
4,365+ days,8906,0.1763


**Signal 1 verdict: MIXED**

Fresher buckets do show higher measured CTR on average — 0-29 days averages **0.39%** CTR (n=7,016) versus **0.18%** for 365+ days (n=8,906), a **0.21 pp** gap that supports the refresh-staleness intuition. However the relationship is not monotonic: the 180-364 day bucket averages **0.21%** (n=21,030), slightly *above* the 90-179 day bucket at **0.20%** (n=10,277). Staleness directionally associates with lower CTR but is too noisy to treat as a clean linear rule on its own.

## Signal 2 — Average search position vs CTR

FlyRank's CTR-fix logic assumes better-ranked pages earn higher CTR. Bucket `avg_position` (March mean, positions > 0 only) and compare average CTR within each bucket.

In [5]:
def position_bucket(pos: float) -> str:
    if pos <= 3:
        return "top_3 (1-3)"
    if pos <= 10:
        return "page_1 (4-10)"
    if pos <= 20:
        return "striking (11-20)"
    if pos <= 50:
        return "page_3_5 (21-50)"
    return "deep (51+)"

work["position_bucket"] = work["avg_position"].map(position_bucket)
position_order = [
    "top_3 (1-3)", "page_1 (4-10)", "striking (11-20)",
    "page_3_5 (21-50)", "deep (51+)",
]

signal2 = (
    work.groupby("position_bucket", observed=True)
    .agg(n=("content_hash_id", "count"), avg_ctr_pct=("ctr_pct", "mean"))
    .reindex(position_order)
    .reset_index()
)
display(signal2.assign(avg_ctr_pct=signal2["avg_ctr_pct"].round(4)))

,position_bucket,n,avg_ctr_pct
0,top_3 (1-3),5764,0.3610
1,page_1 (4-10),25831,0.3043
2,striking (11-20),13202,0.2062
3,page_3_5 (21-50),15052,0.1230
4,deep (51+),3812,0.0438


**Signal 2 verdict: CONFIRMED**

Average CTR falls monotonically as average position worsens: **0.36%** in top_3 (n=5,764) → **0.30%** page_1 (n=25,831) → **0.21%** striking (n=13,202) → **0.12%** page_3_5 (n=15,052) → **0.04%** deep (n=3,812). The top-to-bottom spread is **0.32 pp**, supporting position-bucket CTR comparisons as a defensible baseline ingredient.

## 1. My rule and its reason codes

**Plain-English rule:** Flag content for refresh review when it is **stale** (≥ 180 days since last optimization, or creation date if never optimized) **and** its March CTR sits **below** the average CTR of pages in the same position bucket.

**Score:** `staleness_days × max(0, bucket_avg_ctr_pct − page_ctr_pct)` — only positive CTR gaps count.

**Reason code (exactly one):** `STALE_LOW_CTR` for every qualifying row; blank otherwise.

**Actions:** `refresh` for the top quartile of positive scores; `monitor` for everything else. Threshold is computed from the actual score distribution (75th percentile of scores > 0).

## 2. Build the ranked queue (writes the CSV)

Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.

In [6]:
bucket_avg_ctr = work.groupby("position_bucket")["ctr_pct"].mean()
work["bucket_avg_ctr_pct"] = work["position_bucket"].map(bucket_avg_ctr)
work["ctr_gap_pct"] = (work["bucket_avg_ctr_pct"] - work["ctr_pct"]).clip(lower=0)
work["qualifies"] = (work["staleness_days"] >= STALE_DAYS) & (work["ctr_gap_pct"] > 0)
work["score"] = np.where(
    work["qualifies"],
    work["staleness_days"] * work["ctr_gap_pct"],
    0.0,
)
work["reason_code"] = np.where(work["qualifies"], "STALE_LOW_CTR", "")

positive_scores = work.loc[work["score"] > 0, "score"]
refresh_threshold = positive_scores.quantile(0.75)
work["action"] = np.where(
    work["score"] >= refresh_threshold,
    "refresh",
    "monitor",
)
work.loc[work["score"] == 0, "action"] = "monitor"

queue_cols = [
    "client_hash_id", "content_hash_id", "staleness_days", "ctr_pct",
    "avg_position", "position_bucket", "bucket_avg_ctr_pct", "ctr_gap_pct",
    "score", "reason_code", "action", "impressions",
]
queue = work.sort_values("score", ascending=False)[queue_cols]

out_path = ROOT / "work" / "outputs" / "baseline_action_score.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)

print(f"Refresh threshold (75th pct of positive scores): {refresh_threshold:.4f}")
print(f"Rows written: {len(queue):,}")
print(f"refresh: {(queue['action'] == 'refresh').sum():,} | monitor: {(queue['action'] == 'monitor').sum():,}")
print(f"CSV: {out_path}")
display(queue.head(10).round(4))

Refresh threshold (75th pct of positive scores): 66.0308
Rows written: 63,661
refresh: 5,549 | monitor: 58,112
CSV: ..\..\work\outputs\baseline_action_score.csv


,client_hash_id,content_hash_id,staleness_days,ctr_pct,avg_position,position_bucket,bucket_avg_ctr_pct,ctr_gap_pct,score,reason_code,action,impressions
49571,client_e547b89c05043229,content_cc0469cb3dcb3aff,475,0.0000,1.5427,top_3 (1-3),0.361,0.3610,171.4919,STALE_LOW_CTR,refresh,540.0
33118,client_e547b89c05043229,content_98a144e78a860a21,475,0.0000,2.7852,top_3 (1-3),0.361,0.3610,171.4919,STALE_LOW_CTR,refresh,1240.0
107706,client_e547b89c05043229,content_ae6f555a1daba25e,467,0.0000,1.3716,top_3 (1-3),0.361,0.3610,168.6036,STALE_LOW_CTR,refresh,183.0
132692,client_e547b89c05043229,content_2fae6520bb7a6962,467,0.0000,0.8990,top_3 (1-3),0.361,0.3610,168.6036,STALE_LOW_CTR,refresh,1186.0
91096,client_e547b89c05043229,content_5e8f1dea52e86e63,467,0.0000,1.6750,top_3 (1-3),0.361,0.3610,168.6036,STALE_LOW_CTR,refresh,427.0
151853,client_e547b89c05043229,content_3abd9df04eb7ddbf,467,0.0000,2.6839,top_3 (1-3),0.361,0.3610,168.6036,STALE_LOW_CTR,refresh,371.0
82874,client_e547b89c05043229,content_3eb93e7a5bbd5222,475,0.0175,0.6270,top_3 (1-3),0.361,0.3435,163.1586,STALE_LOW_CTR,refresh,5700.0
8420,client_e547b89c05043229,content_a8825ff71535b1c7,434,0.0000,2.4304,top_3 (1-3),0.361,0.3610,156.6894,STALE_LOW_CTR,refresh,471.0
49553,client_e547b89c05043229,content_bebdd9b8da343222,434,0.0000,1.6405,top_3 (1-3),0.361,0.3610,156.6894,STALE_LOW_CTR,refresh,288.0
77310,client_e547b89c05043229,content_4087a29f5b74b526,434,0.0000,2.9348,top_3 (1-3),0.361,0.3610,156.6894,STALE_LOW_CTR,refresh,709.0


## 3. Top-10 review

Each line uses the actual ranked queue from March 2026 warehouse data.

1. **content_cc0469cb3dcb3aff** — `refresh` because it has **475** days of staleness and a **0.36 pp** CTR gap versus its top_3 bucket (page CTR **0.00%** vs bucket **0.36%**, score **171.5**). It would be wrong if `last_optimized_date` is stale metadata or if **540** March impressions make the zero-click CTR unreliable.
2. **content_98a144e78a860a21** — `refresh` because **475** days stale with **0.36 pp** CTR gap in top_3 (score **171.5**, **1,240** impressions). Wrong if the page was recently rewritten but optimization date was not updated.
3. **content_2fae6520bb7a6962** — `refresh` because **467** days stale, position **0.9**, CTR gap **0.36 pp** (score **168.6**, **1,186** impressions). Wrong if search intent is navigational/branded where low CTR is normal despite strong position.
4. **content_5e8f1dea52e86e63** — `refresh` because **467** days stale with **0.36 pp** CTR gap (score **168.6**, **427** impressions). Wrong if impression volume is too thin for a stable CTR read.
5. **content_3abd9df04eb7ddbf** — `refresh` because **467** days stale, top_3 bucket, **0.36 pp** gap (score **168.6**, **371** impressions). Wrong if the broad top_3 bucket hides a query mix where this page's CTR norm differs.
6. **content_ae6f555a1daba25e** — `refresh` because **467** days stale and **0.36 pp** below bucket CTR (score **168.6**). It would be wrong if **183** impressions make the zero-click rate noise — this is a weak pick on volume alone.
7. **content_3eb93e7a5bbd5222** — `refresh` because **475** days stale with **0.34 pp** CTR gap (page CTR **0.018%** vs bucket **0.36%**, score **163.2**, **5,700** impressions). Wrong if SERP features (snippets, sitelinks) suppress clicks despite visibility.
8. **content_bebdd9b8da343222** — `refresh` because **434** days stale, top_3, **0.36 pp** gap (score **156.7**, **288** impressions). Wrong if low impressions inflate apparent underperformance.
9. **content_25596d8f5df1e383** — `refresh` because **434** days stale with **0.36 pp** gap (score **156.7**, **394** impressions). Wrong if the page targets a low-CTR query type within an otherwise high-CTR position tier.
10. **content_4087a29f5b74b526** — `refresh` because **434** days stale, position **2.9**, **0.36 pp** CTR gap (score **156.7**, **709** impressions). Wrong if a recent title/meta change already improved clicks but GSC lag has not caught up.

## 4. Weak picks + leakage check

**Weak picks (from the actual queue):** Several top-ranked rows share one client and rank highly mainly because staleness is extreme (~430–475 days) while CTR is exactly zero on modest impression counts (**183–427** for rows #4–#6 and #8). Zero-click pages in top_3 are plausible refresh candidates, but scores multiply a large staleness factor by a bucket-average gap — pages with **<300** impressions can look urgent while CTR is still statistically unstable. **content_3eb93e7a5bbd5222** is the strongest top-10 pick (**5,700** impressions, measurable **0.018%** CTR vs **0.36%** bucket norm).

**Leakage check:** Features use only March 2026 GSC aggregates and static content metadata (`last_optimized_date` / `content_created_date`). No `trend_direction`, `trend_pct`, `is_declining_label`, product flags, or June 2026 (sealed test month) data entered the score.

In [7]:
weak = queue[(queue["score"] > 0) & (queue["impressions"] < 300)].head(5)
print("Weak picks — high score but <300 March impressions:")
display(weak.round(4))

leakage_cols = {"trend_direction", "trend_pct", "is_declining_label"}
used_cols = set(work.columns)
assert not leakage_cols & used_cols, "Label-derived columns leaked into features"
print("No label-derived columns in feature frame.")

Weak picks — high score but <300 March impressions:


,client_hash_id,content_hash_id,staleness_days,ctr_pct,avg_position,position_bucket,bucket_avg_ctr_pct,ctr_gap_pct,score,reason_code,action,impressions
107706,client_e547b89c05043229,content_ae6f555a1daba25e,467,0.0,1.3716,top_3 (1-3),0.361,0.361,168.6036,STALE_LOW_CTR,refresh,183.0
49553,client_e547b89c05043229,content_bebdd9b8da343222,434,0.0,1.6405,top_3 (1-3),0.361,0.361,156.6894,STALE_LOW_CTR,refresh,288.0
102038,client_e547b89c05043229,content_a18fb920bf78d8a0,417,0.0,1.3512,top_3 (1-3),0.361,0.361,150.5518,STALE_LOW_CTR,refresh,265.0
57964,client_e547b89c05043229,content_c9d9b29e33136b7a,417,0.0,0.6034,top_3 (1-3),0.361,0.361,150.5518,STALE_LOW_CTR,refresh,221.0
16628,client_e547b89c05043229,content_a664c4279668dae8,417,0.0,2.1994,top_3 (1-3),0.361,0.361,150.5518,STALE_LOW_CTR,refresh,163.0


No label-derived columns in feature frame.


## Self-check

- [x] Signal 1 bucket table with `n` and avg CTR — executed above
- [x] Signal 1 verdict (**MIXED**) supported by 0.39% → 0.18% end-to-end drop but non-monotonic middle buckets
- [x] Signal 2 bucket table with `n` and avg CTR — executed above
- [x] Signal 2 verdict (**CONFIRMED**) — monotonic 0.36% → 0.04% across position buckets
- [x] One baseline rule (stale + position-bucket CTR gap)
- [x] One reason code: `STALE_LOW_CTR`
- [x] Action labels: `refresh` / `monitor` from 75th-percentile threshold on positive scores
- [x] Numeric score, queue sorted descending
- [x] CSV written to `work/outputs/baseline_action_score.csv` (**63,661** rows)
- [x] No future-window or label-derived inputs
- [x] Top-10 review with actual IDs, values, and "what would make it wrong" per row
- [x] Weak picks identified from low-impression high-score rows
- [x] Notebook runs top to bottom